In [1]:
!pip install -q datasets evaluate
!pip install -q evaluate

In [2]:
from datasets import load_dataset

ds = load_dataset("thainq107/abte-restaurants")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/454 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/61.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3602 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1119 [00:00<?, ? examples/s]

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
def tokenize_and_align_labels(examples):
    sentences, sentence_tags = [], []
    labels = []
    for tokens, pols in zip(examples['Tokens'], examples['Polarities']):
        bert_tokens = []
        bert_att = []
        pols_label = 0
        for i in range(len(tokens)):
            t = tokenizer.tokenize(tokens[i])
            bert_tokens += t
            if int(pols[i]) != -1:   # -1 = không phải aspect
                bert_att += t
                pols_label = int(pols[i])

        sentences.append(" ".join(bert_tokens))
        sentence_tags.append(" ".join(bert_att))
        labels.append(pols_label)

    tokenized_inputs = tokenizer(
        sentences, sentence_tags,
        padding=True, truncation=True, return_tensors="pt"
    )
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

preprocessed_ds = ds.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/3602 [00:00<?, ? examples/s]

Map:   0%|          | 0/1119 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoModelForSequenceClassification

# 3 nhãn: 0=Negative, 1=Neutral, 2=Positive
id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="absa-restaurants-albert-base-v2",
    learning_rate=2e-5, # Tốc độ học — kiểm soát mỗi bước update trọng số thay đổi bao nhiêu. 2e-5 = 0.00002, đây là giá trị chuẩn cho fine-tuning BERT/ALBERT — đủ nhỏ để không phá vỡ trọng số pretrained.
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=50,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy", # Chỉ định chỉ số nào dùng để xác định "best checkpoint" khi load_best_model_at_end=True. Ở đây dùng accuracy thay vì mặc định (eval_loss).
)

```
LR quá lớn → model "nhảy loạn", không hội tụ
LR quá nhỏ → train rất chậm, dễ mắc kẹt local minima
2e-5 ✅ → sweet spot cho fine-tuning Transformer
```

In [7]:
import numpy as np
import huggingface_hub

# Apply the patch before importing evaluate
try:
    from huggingface_hub.utils import HfFolder
    import huggingface_hub.hf_api
    huggingface_hub.hf_api.HfFolder = HfFolder
except ImportError:
    pass

import evaluate
from transformers import Trainer

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred # ① Unpack
    predictions = np.argmax(logits, axis=-1) # ② Decode
    return metric.compute(predictions=predictions, references=labels) # ③ Tính

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=preprocessed_ds["train"],
    eval_dataset=preprocessed_ds["test"], # ⚠️ Dùng "test" làm eval
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.773532,0.649687
2,No log,0.589942,0.755139
3,No log,0.527090,0.790885
4,No log,0.519357,0.799821
5,No log,0.491162,0.821269
6,No log,0.486713,0.816801
7,No log,0.570556,0.807864
8,No log,0.588303,0.810545
9,No log,0.627330,0.812332
10,No log,0.613777,0.815907


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1450, training_loss=0.1141265895448882, metrics={'train_runtime': 3456.2965, 'train_samples_per_second': 52.108, 'train_steps_per_second': 0.42, 'total_flos': 6896396457309600.0, 'train_loss': 0.1141265895448882, 'epoch': 50.0})



```
① Unpack eval_pred
Trainer tự động truyền vào một tuple (logits, labels):
```



```
logits  → shape (N, num_classes) — điểm số thô của model
labels  → shape (N,)             — nhãn đúng
```





```
② np.argmax(logits, axis=-1)
Chuyển logits → class dự đoán bằng cách lấy index có giá trị lớn nhất theo chiều class:
```


```
logits = [[-1.2,  3.5,  0.8],   →  argmax  →  predictions = [1, 2, 0]
          [ 0.1,  0.3,  4.1],
          [ 2.9, -0.5,  1.1]]
```






```
③ metric.compute(...)
So sánh predictions vs references (labels đúng) và trả về:
```


```
{"accuracy": 0.91}  # ví dụ
```




In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

In [9]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
atsc_model = AutoModelForSequenceClassification.from_pretrained(
    "/content/absa-restaurants-albert-base-v2/checkpoint-1450"
)
id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [10]:
def absa_inference(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = atsc_model(**inputs)
        logits = outputs.logits
        predicted_class = np.argmax(logits.detach().cpu().numpy())
        sentiment = id2label[predicted_class]
        score = torch.softmax(logits, dim=1)[0][predicted_class].item()
    return {"text": text, "sentiment": sentiment, "score": score}

In [11]:
text = "The price was too high, but the cab was amazing."
result = absa_inference(text)
print(result)
# Output: {'text': '...', 'sentiment': 'Positive', 'score': 0.9996}

{'text': 'The price was too high, but the cab was amazing.', 'sentiment': 'Positive', 'score': 0.9981959462165833}
